In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [25]:
url = "https://raw.githubusercontent.com/THGLab/LP-PDBBind/refs/heads/master/dataset/LP_PDBBind.csv"
lp_df = pd.read_csv(url)
lp_df.head()

,Unnamed: 0,header,smiles,category,seq,resolution,date,type,new_split,CL1,CL2,CL3,remove_for_balancing_val,kd/ki,value,covalent
0,6r8o,isomerase,CSc1ccccc1[C@H]1CCCN1C(=O)CNC(=O)NCc1ccc2c(c1)...,refined,GNPLVYLDVDANGKPLGRVVLELKADVVPKTAENFRALCTGEKGFG...,1.36,2019-11-27,isomerase,test,True,True,True,False,Kd=0.006uM,8.22,False
1,3fh7,hydrolase/hydrolase inhibitor,O=C([O-])CCC[N@H+]1CCC[C@H]1COc1ccc(Oc2ccc(Cl)...,refined,VDTCSLASPASVCRTKHLHLRCSVDFTRRTLTGTAALTVQSQEDNL...,2.05,2010-01-05,hydrolase,test,True,True,True,False,Kd=25nM,7.60,False
2,4b7r,hydrolase,CCC(CC)O[C@@H]1C[C@H](C(=O)[O-])C[C@H]([NH3+])...,refined,VKLAGNSSLCPVSGWAIYSKDNSVRIGSKGDVFVIREPFISCSPLE...,1.90,2012-10-03,hydrolase,NaN,True,True,True,False,Ki=0.23nM,9.64,False
3,3qfd,immune system,CC[C@H](C)[C@H](NC(=O)CNC(=O)[C@H](C)NC(=O)[C@...,refined,GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...,1.68,2011-09-28,other,train,False,False,False,False,Kd=68uM,4.17,False
4,3fvn,membrane protein,[NH3+][C@@H](C[C@]1(C(=O)[O-])C[C@H]2OCC[C@@H]...,refined,ANRTLIVTTILEEPYVMYRKSDKPLYGNDRFEGYCLDLLKELSNIL...,1.50,2010-01-19,membrane,val,True,True,True,False,Ki=169nM,6.77,False


In [26]:
import re

def parse_kd_ki(kd_ki_str):

    if pd.isna(kd_ki_str):
        return np.nan
    
    match = re.match(r'(Kd|Ki)=([0-9.]+)([un]M)', kd_ki_str)
    if not match:
        return np.nan
    
    _, value_str, unit = match.groups()
    value = float(value_str)
    
    if unit == 'nM':
        value /= 1000
    
    return value

lp_df['IC50_uM'] = lp_df['kd/ki'].apply(parse_kd_ki)

lp_df = lp_df[lp_df['IC50_uM'] > 0]

# Вычисляем pIC50
lp_df['pIC50'] = -np.log10(lp_df['IC50_uM'] * 1e-6)  # µM → M

print("\nСтатистика по pIC50:")
print(lp_df['pIC50'].describe())


Статистика по pIC50:
count    11056.000000
mean         6.338286
std          1.669019
min          1.823909
25%          5.050610
50%          6.292771
75%          7.552842
max         11.920819
Name: pIC50, dtype: float64


In [27]:
print(lp_df.columns)

Index(['Unnamed: 0', 'header', 'smiles', 'category', 'seq', 'resolution',
       'date', 'type', 'new_split', 'CL1', 'CL2', 'CL3',
       'remove_for_balancing_val', 'kd/ki', 'value', 'covalent', 'IC50_uM',
       'pIC50'],
      dtype='object')


In [30]:
print(long_df.columns)

Index(['Ligand', 'Protein', 'Residue', 'InteractionType'], dtype='object')


In [29]:
df = pd.read_csv("protein_rightligand_interaction2.0.csv")
residue_columns = [col for col in df.columns if col not in ['Ligand', 'Protein']]

long_df = df.melt(
    id_vars=['Ligand', 'Protein'],
    value_vars=residue_columns,
    var_name='Residue',
    value_name='InteractionType'
)

long_df = long_df[long_df['InteractionType'] != "0"].reset_index(drop=True)
long_df['Protein'] = long_df['Protein'].str.lower()
print(f"Всего взаимодействий: {len(long_df)}")
print(long_df.head())

Всего взаимодействий: 526
  Ligand Protein    Residue InteractionType
0    T20    4rmz  ALA_A:211     hydrophobic
1    XPY    4y73  ALA_A:211     hydrophobic
2    4GD    4yo6  ALA_A:211     hydrophobic
3    4GF    4yp8  ALA_A:211     hydrophobic
4    4S1    4ztl  ALA_A:211     hydrophobic


In [34]:
# В long_df: Protein — уже PDB ID в нижнем регистре (как '4rmz')
long_df['pdb_key'] = long_df['Protein'].str.lower()

# В lp_df: PDB ID находится в 'Unnamed: 0'
lp_df['pdb_key'] = lp_df['Unnamed: 0'].str.lower()

# Проверим пересечение
common = set(long_df['pdb_key']) & set(lp_df['pdb_key'])
print(f"Общих PDB ID: {len(common)}")
if common:
    print("Примеры:", sorted(list(common))[:5])

# Теперь объединяем
merged = pd.merge(
    long_df[['Ligand', 'Protein', 'Residue', 'InteractionType', 'pdb_key']],
    lp_df[['pdb_key', 'pIC50']],
    on='pdb_key',
    how='inner'
).drop(columns='pdb_key')

print("Результат:", merged.shape)

Общих PDB ID: 5
Примеры: ['6mom', '6o8u', '6o94', '6o95', '6o9d']
Результат: (47, 5)


In [33]:
print("Примеры Protein из long_df:")
print(long_df['Protein'].head().tolist())

print("Первые 10 значений Unnamed: 0:")
print(lp_df['Unnamed: 0'].head(10).tolist())

Примеры Protein из long_df:
['4rmz', '4y73', '4yo6', '4yp8', '4ztl']
Первые 10 значений Unnamed: 0:
['6r8o', '3fh7', '4b7r', '3qfd', '3fvn', '1h6h', '5iwg', '6ml9', '3nq3', '6mle']


In [37]:
values = df['Ligand'].tolist()
print(values)

['T20', 'STU', '42P', 'XPY', '4GD', '4GF', '4S1', '4S2', '4S3', '6QY', '6QX', '6R1', '6R0', '6QZ', '6YD', '6YE', '76Q', '76P', '8BV', '8BY', '8C1', '8CD', '8CG', '9YY', '9YS', '0LI', 'J8A', 'J87', 'DL1', 'CJT', 'CJQ', 'CJN', 'CKN', 'EXF', 'JX4', 'KFD', 'LS7', 'LRS', 'LSV', 'LTY', 'K1H', 'K1E', 'NBK', 'N9Z', 'NB5', 'NBW', 'ND2', 'QL7', 'R7S', 'FJ0', 'FJ9', 'B8I', 'B6I', 'B4U', 'B7R', 'SO9', 'ZVD', 'ZVG', 'ZVA', 'VDC', 'VE9', 'VEU', 'X1T', 'WFQ', 'Y9T', 'YJU', 'YK0', 'Y9T']
